Bibliotecas necessarias

In [1]:
import pandas as pd

import win32com.client as client

import datetime as dt

import os

from dotenv import load_dotenv

Acessando as variáveis de ambientes

In [2]:
load_dotenv()

EMAIL = os.getenv('EMAIL')

lendo os dados do banco de dados (JSON)

In [3]:
df_dados = pd.read_json("https://alarme-b3d19-default-rtdb.firebaseio.com/.json", orient='index')

df_dados['Data'] = pd.to_datetime(df_dados['Data'])

display(df_dados)

,Data,Vazao
-OtoU7mjE2h37dyAN9Lq,2026-05-29 13:06:42.099363,117
-OtoUAl40L_nPROe0ura,2026-05-29 13:06:53.314368,124
-OtoUDZTNuVvJvqSCKV3,2026-05-29 13:07:05.489741,159
-OtoUGSQfQjIx4OwS30u,2026-05-29 13:07:16.972128,199
-OtoUJM3z-Uloy3byHAG,2026-05-29 13:07:28.809154,168
...,...,...
-OtpQ5FVgCZqCHkfcgkC,2026-05-29 17:28:40.271307,112
-OtpQ7sBX8lHSzedTQO_,2026-05-29 17:28:51.511296,182
-OtpQqktlzhl58xEiszu,2026-05-29 17:31:55.362165,126
-OtpQtMhT2u4Jcfs9s5H,2026-05-29 17:32:10.190601,125


Estipulando o periodo de análise

In [4]:
agora = dt.datetime.now()

delta = dt.timedelta(minutes=2)

tempo_limite = agora - delta

In [5]:
dados_analise = df_dados.loc[df_dados['Data'] > tempo_limite] 

display(dados_analise)

,Data,Vazao
-OtpQqktlzhl58xEiszu,2026-05-29 17:31:55.362165,126
-OtpQtMhT2u4Jcfs9s5H,2026-05-29 17:32:10.190601,125
-OtpQvyehnjtc-6NBV-R,2026-05-29 17:32:20.860718,178


Filtrando dados na base

In [6]:
aviso = dados_analise.loc[dados_analise['Vazao'] > 150]

display(aviso)

,Data,Vazao
-OtpQvyehnjtc-6NBV-R,2026-05-29 17:32:20.860718,178


In [7]:
lista_avisos = aviso[['Data', 'Vazao']].values.tolist()

print(lista_avisos)

[[Timestamp('2026-05-29 17:32:20.860718'), 178]]


Criando o e-mail

In [8]:
outlook = client.Dispatch('Outlook.Application')

In [ ]:
if len(lista_avisos) == 0:
    
    print("A lista está vazia, não há valores para enviar")

else:
    
    for avisos in lista_avisos:
    
        data_aviso = avisos[0]
    
        data_aviso = data_aviso.strftime("%d/%m/%Y %H:%M:%S")
    
        vazao_aviso = avisos[1]
    
        mensagem = outlook.CreateItem(0)

        mensagem.Display()

        mensagem.To = EMAIL

        mensagem.Subject = "Alerta, valores abaixo de 150"

        mensagem.Body = f'''Prezado, no dia / hora {data_aviso} o valor da vazão foi de {vazao_aviso}, superando o valor aceitável'''

        mensagem.save()

        mensagem.Send()